<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 1. Aspectos generales

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Implementación y verificación de la transición **Raw → Bronze → Silver** bajo Arquitectura Medallón en PySpark
para los tres dominios del Examen Final:

- **Dominio 1 (SIAF-MEF 2009–2026):** 20 archivos CSV de ejecución de ingresos (16,719,553 filas brutas; 15,185,314 netas).
- **Dominio 2 (SISMEPRE / ATM):** 7 tablas relacionales de metas prediales y cuestionarios (493,558 filas totales).
- **Dominio 3 (RENAMU 2022 / INEI):** Censo Nacional de Municipalidades completo (`Base_RENAMU_2022_f.csv`, 1,874 filas y 1,368 variables).

In [1]:
import csv
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ByteType, DecimalType, IntegerType, LongType,
    ShortType, StringType, StructField, StructType,
)

In [2]:
# Inicializar SparkSession replicando exactamente la configuracion del curso
spark = (
    SparkSession.builder
    .appName("MEF_Bronze_to_Silver")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.memory.fraction", "0.8")
    .getOrCreate()
)

spark.sparkContext.setCheckpointDir("/tmp/spark_checkpoint")
spark.sparkContext.setLogLevel("WARN")
spark

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 2. Rutas de capas y esquema explicito StructType (sin inferSchema)

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Define las rutas de la arquitectura Medallón y demuestra la política de ingesta:

- **Prohibido `inferSchema=True` en archivos CSV:** todas las columnas se definen explícitamente como `StringType()`
  mediante `StructType` para proteger ceros a la izquierda en `DEPARTAMENTO_EJECUTORA`, `PROVINCIA_EJECUTORA`,
  `DISTRITO_EJECUTORA`, `EJECUTORA` y `Ubigeo`.
- **Remediación en `rentas_respuestas.csv`:** tokenización posicional (`parts[:6]`, `parts[6:-4]`, `parts[-4:]`) para recuperar
  las 9 filas con comillas dobles internas sin escapar sin perder ninguno de los 322,675 registros.
- **Remediación en `Base_RENAMU_2022_f.csv`:** lectura con separador `;` y eliminación del carácter BOM `\ufeff` (`utf-8-sig`).

In [3]:
RAW    = "/home/jovyan/Final/proyecto_final/raw"
BRONZE = "/home/jovyan/Final/proyecto_final/bronze"
SILVER = "/home/jovyan/Final/proyecto_final/silver"
GOLD   = "/home/jovyan/Final/proyecto_final/gold"

def str_schema(columns):
    return StructType([StructField(c, StringType(), True) for c in columns])

SIAF_COLS = [
    "ANO_DOC", "MES_DOC", "NIVEL_GOBIERNO", "NIVEL_GOBIERNO_NOMBRE",
    "SECTOR", "SECTOR_NOMBRE", "PLIEGO", "PLIEGO_NOMBRE",
    "SEC_EJEC", "EJECUTORA", "EJECUTORA_NOMBRE",
    "DEPARTAMENTO_EJECUTORA", "DEPARTAMENTO_EJECUTORA_NOMBRE",
    "PROVINCIA_EJECUTORA", "PROVINCIA_EJECUTORA_NOMBRE",
    "DISTRITO_EJECUTORA", "DISTRITO_EJECUTORA_NOMBRE",
    "FUENTE_FINANCIAMIENTO", "FUENTE_FINANCIAMIENTO_NOMBRE",
    "RUBRO", "RUBRO_NOMBRE", "TIPO_RECURSO", "TIPO_RECURSO_NOMBRE",
    "GENERICA", "GENERICA_NOMBRE", "SUBGENERICA", "SUBGENERICA_NOMBRE",
    "SUBGENERICA_DET", "SUBGENERICA_DET_NOMBRE",
    "ESPECIFICA", "ESPECIFICA_NOMBRE", "ESPECIFICA_DET", "ESPECIFICA_DET_NOMBRE",
    "MONTO_PIA", "MONTO_PIM", "MONTO_RECAUDADO",
]
SIAF_SCHEMA = str_schema(SIAF_COLS)
print(f"Esquema SIAF explícito definido: {len(SIAF_SCHEMA)} columnas StringType")

# Verificación de encabezado RENAMU limpiando BOM utf-8-sig
with open(f"{RAW}/renamu/Base_RENAMU_2022_f.csv", "r", encoding="utf-8-sig") as f:
    renamu_headers = [h.lstrip("\ufeff") for h in next(csv.reader(f, delimiter=";"))]
print(f"Encabezado RENAMU 2022: {len(renamu_headers)} columnas | Col[0]={repr(renamu_headers[0])} | Col[5]={repr(renamu_headers[5])}")

# Verificación en vivo de tokenización posicional sobre rentas_respuestas.csv crudo
df_resp_txt = (
    spark.read.text(f"{RAW}/sismepre/rentas_respuestas.csv")
    .filter(~F.col("value").startswith('"SEC_EJEC"'))
    .filter(F.length(F.trim(F.col("value"))) > 0)
)
stripped = F.regexp_replace(F.col("value"), r'^"|"\r?$', "")
parts = F.split(stripped, '","')
n_parts = F.size(parts)
df_resp_parsed = df_resp_txt.select(
    parts[0].alias("SEC_EJEC"),
    parts[1].alias("ANO_APLICACION"),
    F.regexp_replace(
        F.when(n_parts > 11, F.concat_ws('","', F.slice(parts, 7, n_parts - 10))).otherwise(parts[6]),
        r'"', "'"
    ).alias("RESPUESTA_TEXTO"),
    F.element_at(parts, -1).alias("ESTADO_REGISTRO"),
)
print(f"Registros recuperados en rentas_respuestas.csv con tokenización posicional: {df_resp_parsed.count():,}")

Esquema SIAF explícito definido: 36 columnas StringType
Encabezado RENAMU 2022: 1368 columnas | Col[0]='Año' | Col[5]='Ubigeo'
Registros recuperados en rentas_respuestas.csv con tokenización posicional: 322,675


<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 3. Lectura y verificacion de la capa Bronze

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Lee los datasets persistidos en `bronze/` en formato Parquet comprimido (Snappy) con la columna `_archivo_origen`
y valida los conteos exactos de ingesta y la preservación íntegra de los ceros a la izquierda.

In [4]:
%%time
bronze_siaf  = spark.read.parquet(f"{BRONZE}/siaf")
bronze_esat  = spark.read.parquet(f"{BRONZE}/sismepre/rentas_esat_estadistica_atm")
bronze_resp  = spark.read.parquet(f"{BRONZE}/sismepre/rentas_respuestas")
bronze_form  = spark.read.parquet(f"{BRONZE}/sismepre/rentas_formulario")
bronze_preg  = spark.read.parquet(f"{BRONZE}/sismepre/rentas_preguntas")
bronze_est   = spark.read.parquet(f"{BRONZE}/sismepre/rentas_entidad_estado")
bronze_ren   = spark.read.parquet(f"{BRONZE}/renamu/renamu_2022")

print(f"bronze/siaf                         : {bronze_siaf.count():>12,} filas  ({len(bronze_siaf.columns)} cols)")
print(f"bronze/sismepre/rentas_esat         : {bronze_esat.count():>12,} filas  ({len(bronze_esat.columns)} cols)")
print(f"bronze/sismepre/rentas_respuestas   : {bronze_resp.count():>12,} filas  ({len(bronze_resp.columns)} cols)")
print(f"bronze/sismepre/rentas_entidad_est  : {bronze_est.count():>12,} filas  ({len(bronze_est.columns)} cols)")
print(f"bronze/sismepre/rentas_formulario   : {bronze_form.count():>12,} filas  ({len(bronze_form.columns)} cols)")
print(f"bronze/sismepre/rentas_preguntas    : {bronze_preg.count():>12,} filas  ({len(bronze_preg.columns)} cols)")
print(f"bronze/renamu/renamu_2022           : {bronze_ren.count():>12,} filas  ({len(bronze_ren.columns)} cols)")

# Verificar preservacion de ceros a la izquierda en Bronze (texto)
ceros_prov = bronze_siaf.filter(F.col("PROVINCIA_EJECUTORA").startswith("0")).count()
ceros_dist = bronze_siaf.filter(F.col("DISTRITO_EJECUTORA").startswith("0")).count()
print(f"\nRegistros con cero inicial preservado en Bronze -> PROVINCIA: {ceros_prov:,} | DISTRITO: {ceros_dist:,}")

bronze/siaf                         :   16,719,553 filas  (37 cols)
bronze/sismepre/rentas_esat         :      149,298 filas  (43 cols)
bronze/sismepre/rentas_respuestas   :      322,675 filas  (12 cols)
bronze/sismepre/rentas_entidad_est  :       20,391 filas  (14 cols)
bronze/sismepre/rentas_formulario   :           98 filas  (11 cols)
bronze/sismepre/rentas_preguntas    :          836 filas  (16 cols)
bronze/renamu/renamu_2022           :        1,874 filas  (1369 cols)

Registros con cero inicial preservado en Bronze -> PROVINCIA: 14,792,541 | DISTRITO: 13,043,914
CPU times: user 34.2 ms, sys: 32.4 ms, total: 66.6 ms
Wall time: 16 s


<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 4. Inspeccion de esquemas y muestras en capa Bronze

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Revisión de los esquemas en Bronze (100% `string` + `_archivo_origen`) y primeras 5 filas de cada dominio.

In [5]:
print("--- Schema bronze/siaf ---")
bronze_siaf.printSchema()

print("--- Muestra bronze/siaf (columnas clave) ---")
bronze_siaf.select(
    "ANO_DOC", "MES_DOC", "NIVEL_GOBIERNO", "SEC_EJEC", "EJECUTORA",
    "DEPARTAMENTO_EJECUTORA", "PROVINCIA_EJECUTORA", "DISTRITO_EJECUTORA",
    "RUBRO", "MONTO_PIA", "MONTO_PIM", "MONTO_RECAUDADO"
).show(5, truncate=False)

print("--- Muestra bronze/sismepre/rentas_respuestas ---")
bronze_resp.select(
    "SEC_EJEC", "ANO_APLICACION", "FORMULARIO_ID", "PREGUNTA_ID",
    "RESPUESTA_TEXTO", "RESPUESTA_DECIMAL", "RESPUESTA_ENTERO"
).show(5, truncate=True)

print("--- Muestra bronze/renamu/renamu_2022 ---")
bronze_ren.select("Año", "idmunici", "Ubigeo", "Departamento", "Provincia", "Distrito", "Tipomuni", "P32_1_T", "P17_7", "P17_8").show(5, truncate=False)

--- Schema bronze/siaf ---
root
 |-- ANO_DOC: string (nullable = true)
 |-- MES_DOC: string (nullable = true)
 |-- NIVEL_GOBIERNO: string (nullable = true)
 |-- NIVEL_GOBIERNO_NOMBRE: string (nullable = true)
 |-- SECTOR: string (nullable = true)
 |-- SECTOR_NOMBRE: string (nullable = true)
 |-- PLIEGO: string (nullable = true)
 |-- PLIEGO_NOMBRE: string (nullable = true)
 |-- SEC_EJEC: string (nullable = true)
 |-- EJECUTORA: string (nullable = true)
 |-- EJECUTORA_NOMBRE: string (nullable = true)
 |-- DEPARTAMENTO_EJECUTORA: string (nullable = true)
 |-- DEPARTAMENTO_EJECUTORA_NOMBRE: string (nullable = true)
 |-- PROVINCIA_EJECUTORA: string (nullable = true)
 |-- PROVINCIA_EJECUTORA_NOMBRE: string (nullable = true)
 |-- DISTRITO_EJECUTORA: string (nullable = true)
 |-- DISTRITO_EJECUTORA_NOMBRE: string (nullable = true)
 |-- FUENTE_FINANCIAMIENTO: string (nullable = true)
 |-- FUENTE_FINANCIAMIENTO_NOMBRE: string (nullable = true)
 |-- RUBRO: string (nullable = true)
 |-- RUBRO_NOMB

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 5. Reglas de saneamiento y transformacion hacia capa Silver

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Verifica las reglas de limpieza y calidad aplicadas sobre `silver/`:

1. **SIAF:** exclusión de cortes superpuestos (`2025-Ingreso-Mensual.csv` y `2026-Ingreso-Mensual.csv`, `-1,534,239` filas)
   **y filtro estricto municipal `NIVEL_GOBIERNO = 'M'`** (`12,735,873` filas en Silver, reducción de `-23.83%` respecto a `16,719,553` brutas),
   `TRIM` + colapso `\s+` + vacío a `NULL`, `LPAD` a 6 dígitos en `UBIGEO` (`'0'` → `NULL` en departamento),
   casteo a `DecimalType(18, 2)` y preservación de los `72,194` extornos negativos municipales (`ES_EXTORNO_NEGATIVO`).
2. **SISMEPRE / ESAT:** exclusión de `38,749` filas de Arbitrios por `TITULO` (nunca por `FORMULARIO_ID`),
   vigencia por bloque (`VIG_EMISION`, `VIG_RECAUD`, `VIG_SALDO`), reconstrucción de cierre anual `MES_ESTADISTICA = 13` desde 2024
   (`Ind_Cierre_Reconstruido = 1`) y banderas de atípicos (`Ind_Outlier_Saldo`, `Ind_Outlier_Emision`).
3. **RENAMU 2022:** `Ubigeo` a 6 dígitos texto y extracción tipada de variables operativas (`Tipomuni`, `P32_1_T`, `P17_7`, `P17_8`, `P31_3`, `P33A`, `P19D_T`).

In [6]:
%%time
silver_siaf = spark.read.parquet(f"{SILVER}/siaf")
silver_esat = spark.read.parquet(f"{SILVER}/sismepre/rentas_esat_estadistica_atm")
silver_resp = spark.read.parquet(f"{SILVER}/sismepre/rentas_respuestas")
silver_est  = spark.read.parquet(f"{SILVER}/sismepre/rentas_entidad_estado")
silver_ren  = spark.read.parquet(f"{SILVER}/renamu/renamu_2022")

# Verificar exclusion de snapshots 2025/2026 y filtro estricto NIVEL_GOBIERNO='M' en Silver SIAF
n_bronze_siaf = bronze_siaf.count()
df_bn = bronze_siaf.filter(~F.col("_archivo_origen").rlike(r"2025-Ingreso-Mensual\.csv|2026-Ingreso-Mensual\.csv"))
snap_excl = n_bronze_siaf - df_bn.count()
no_muni_excl = df_bn.filter(F.trim(F.col("NIVEL_GOBIERNO")) != "M").count()
n_silver_siaf = silver_siaf.count()
red_pct = (1.0 - n_silver_siaf / n_bronze_siaf) * 100.0
n_extornos = silver_siaf.filter(F.col("ES_EXTORNO_NEGATIVO")).count()
n_sin_monto = silver_siaf.filter(F.col("ES_FILA_SIN_MONTO")).count()
print(f"SIAF Bronze bruto: {n_bronze_siaf:,} | Snapshots excluidos: {snap_excl:,} | Otros niveles (E/R): {no_muni_excl:,} | Silver Municipal ('M'): {n_silver_siaf:,} (-{red_pct:.2f}%)")
print(f"SIAF Flags ('M') -> ES_EXTORNO_NEGATIVO: {n_extornos:,} | ES_FILA_SIN_MONTO: {n_sin_monto:,}")

# Verificar reglas ESAT en Silver
n_vig_emi = silver_esat.filter(F.col("VIG_EMISION") == 1).count()
n_vig_rec = silver_esat.filter(F.col("VIG_RECAUD") == 1).count()
n_vig_sal = silver_esat.filter(F.col("VIG_SALDO") == 1).count()
n_recon   = silver_esat.filter(F.col("Ind_Cierre_Reconstruido") == 1).count()
n_out_e   = silver_esat.filter(F.col("Ind_Outlier_Emision") == 1).count()
n_out_s   = silver_esat.filter(F.col("Ind_Outlier_Saldo") == 1).count()
print(f"ESAT Silver total: {silver_esat.count():,} (incluye {n_recon:,} cierres M1-M12 reconstruidos)")
print(f"ESAT Vigentes -> VIG_EMISION: {n_vig_emi:,} | VIG_RECAUD: {n_vig_rec:,} | VIG_SALDO: {n_vig_sal:,}")
print(f"ESAT Outliers -> Ind_Outlier_Emision: {n_out_e} | Ind_Outlier_Saldo: {n_out_s}")

SIAF Bronze bruto: 16,719,553 | Snapshots excluidos: 1,534,239 | Otros niveles (E/R): 2,449,441 | Silver Municipal ('M'): 12,735,873 (-23.83%)
SIAF Flags ('M') -> ES_EXTORNO_NEGATIVO: 72,194 | ES_FILA_SIN_MONTO: 2,334,146
ESAT Silver total: 111,738 (incluye 1,189 cierres M1-M12 reconstruidos)
ESAT Vigentes -> VIG_EMISION: 6,485 | VIG_RECAUD: 46,544 | VIG_SALDO: 8,175
ESAT Outliers -> Ind_Outlier_Emision: 2 | Ind_Outlier_Saldo: 3
CPU times: user 25.3 ms, sys: 39.2 ms, total: 64.5 ms
Wall time: 22 s


<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 6. Verificacion de conteos, esquemas y muestras en capa Silver

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Inspección final de los DataFrames saneados en `silver/` listos para construir la Constelación de Hechos en Gold.

In [7]:
print("--- Schema silver/siaf ---")
silver_siaf.printSchema()

print("--- Muestra silver/siaf ---")
silver_siaf.select(
    "FECHA", "SEC_EJEC", "UBIGEO", "RUBRO", "TIPO_RECURSO_KEY", "CLASIFICADOR",
    "MONTO_PIA", "MONTO_PIM", "MONTO_RECAUDADO", "ES_EXTORNO_NEGATIVO", "ES_FILA_SIN_MONTO"
).show(5, truncate=False)

print("--- Muestra silver/sismepre/rentas_esat_estadistica_atm (cierre anual) ---")
silver_esat.filter(F.col("MES_ESTADISTICA") == 13).select(
    "SEC_EJEC", "UBIGEO", "ANO_ESTADISTICA", "MES_ESTADISTICA",
    "MON_EMISIONPREDIAL_AFECTO", "MON_RECAUDACTUAL_ORDIN", "MON_SALDOPREDIAL_ORD",
    "VIG_EMISION", "VIG_RECAUD", "VIG_SALDO", "Ind_Cierre_Reconstruido"
).show(5, truncate=False)

print("--- Muestra silver/renamu/renamu_2022 ---")
silver_ren.select(
    "Ubigeo", "Departamento", "Provincia", "Distrito", "Tipomuni",
    "P32_1_T_Personal_Trib", "Personal_Total", "Software_Tributario", "Catastro_Urbano", "Indice_TIC_Rentas"
).show(5, truncate=False)

--- Schema silver/siaf ---
root
 |-- ANO_DOC: short (nullable = true)
 |-- MES_DOC: byte (nullable = true)
 |-- NIVEL_GOBIERNO: string (nullable = true)
 |-- NIVEL_GOBIERNO_NOMBRE: string (nullable = true)
 |-- SECTOR: string (nullable = true)
 |-- SECTOR_NOMBRE: string (nullable = true)
 |-- PLIEGO: string (nullable = true)
 |-- PLIEGO_NOMBRE: string (nullable = true)
 |-- SEC_EJEC: integer (nullable = true)
 |-- EJECUTORA: string (nullable = true)
 |-- EJECUTORA_NOMBRE: string (nullable = true)
 |-- DEPARTAMENTO_EJECUTORA: string (nullable = true)
 |-- DEPARTAMENTO_EJECUTORA_NOMBRE: string (nullable = true)
 |-- PROVINCIA_EJECUTORA: string (nullable = true)
 |-- PROVINCIA_EJECUTORA_NOMBRE: string (nullable = true)
 |-- DISTRITO_EJECUTORA: string (nullable = true)
 |-- DISTRITO_EJECUTORA_NOMBRE: string (nullable = true)
 |-- FUENTE_FINANCIAMIENTO: string (nullable = true)
 |-- FUENTE_FINANCIAMIENTO_NOMBRE: string (nullable = true)
 |-- RUBRO: string (nullable = true)
 |-- RUBRO_NOMBRE

In [8]:
spark.stop()